In [1]:
import sys; sys.path.insert(0, "..")        # let the notebook find src/
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

from src.retrieval import dense_search, bm25_search, hybrid_search, rerank, retrieve, label
print("Retrieval module loaded")

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 18:11:56.352000 58153 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12970.73it/s]
INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:11<00:00, 1.93MiB/s]


Retrieval module loaded


In [2]:
def rank_of(target, docs):
    return next((i for i, d in enumerate(docs, 1) if label(d) == target), None)

q = "My employer has not paid my salary for two months"
print("Article 22 rank in dense top 50:", rank_of("labour_law:22", dense_search(q, k=50)))
print("Article 22 rank in BM25  top 50:", rank_of("labour_law:22", bm25_search(q, k=50)))

q_legal = "employer obligation to pay the worker's wage on time; non-payment of wages"   # same question, law's words
print("\nRewritten in legal terms:")
print("  dense top 5:", [label(d) for d in dense_search(q_legal, k=5)])
print("  BM25  top 5:", [label(d) for d in bm25_search(q_legal, k=5)])

Article 22 rank in dense top 50: 15
Article 22 rank in BM25  top 50: 35

Rewritten in legal terms:
  dense top 5: ['exec_regs:16', 'labour_law:9', 'labour_law:33', 'labour_law:22', 'labour_law:34']
  BM25  top 5: ['exec_regs:16', 'labour_law:39', 'labour_law:51', 'exec_regs:12', 'labour_law:28']


In [3]:
q = "What does Article 51 say?"
cands = hybrid_search(q, top_n=10)
print("Hybrid order :", [label(d) for d in cands[:5]])
print("After rerank :")
for d, s in rerank(q, cands, top_n=5):
    print(f"   {s:.3f}  {label(d):28} {d.page_content.splitlines()[2]!r}")

Hybrid order : ['labour_law_amendment_2022:1', 'labour_law:51', 'exec_regs:38', 'labour_law:51', 'pdpl:23']
After rerank :
   0.872  labour_law:51                'End-of-Service Gratuity for Full-Time Workers'
   0.745  labour_law:51                '6. The foregoing is provided that the total end-of-service gratuity for the foreign Worker does'
   0.542  exec_regs:29                 'Controls for the Deduction from the Workers’ End of Service Pay'
   0.007  exec_regs:38                 'Any provision, contradicting or conflicting with the provisions of this resolution, shall be'
   0.007  labour_law:34                'Absence After Leave'


In [4]:
probes = [
    "How many days of annual leave do I get?",
    "What does Article 51 say?",
    "My employer has not paid my salary for two months",
    "end of service gratuity calculation",
    "Can my company share my personal data with others?",
]
for q in probes:
    print(f"\nQ: {q}")
    print("  hybrid :", [label(d) for d in hybrid_search(q, top_n=3)])
    print("  rerank :", [f"{label(d)} ({s:.2f})" for d, s in retrieve(q, top_n=3)])


Q: How many days of annual leave do I get?
  hybrid : ['labour_law:29', 'labour_law:29', 'exec_regs:18']
  rerank : ['labour_law:29 (0.99)', 'exec_regs:18 (0.87)', 'labour_law:29 (0.84)']

Q: What does Article 51 say?
  hybrid : ['labour_law_amendment_2022:1', 'labour_law:51', 'exec_regs:38']
  rerank : ['labour_law:51 (0.87)', 'labour_law:51 (0.74)', 'exec_regs:29 (0.54)']

Q: My employer has not paid my salary for two months
  hybrid : ['labour_law:9', 'labour_law:30', 'labour_law:40']
  rerank : ['labour_law:9 (0.54)', 'labour_law:47 (0.15)', 'labour_law:30 (0.12)']

Q: end of service gratuity calculation
  hybrid : ['labour_law:51', 'labour_law:51', 'labour_law:52']
  rerank : ['labour_law:51 (1.00)', 'labour_law:51 (1.00)', 'labour_law:52 (0.99)']

Q: Can my company share my personal data with others?
  hybrid : ['pdpl:22', 'pdpl:5', 'pdpl:7']
  rerank : ['pdpl:23 (0.83)', 'pdpl:22 (0.52)', 'pdpl:1 (0.32)']


In [5]:
in_scope = probes + ["How long is the probation period?", "Can my employer keep my passport?"]
out_of_scope = [
    "What is the fee for a UAE Golden Visa?",
    "What is the IRS 401k contribution limit for 2026?",
    "Who won the FIFA World Cup in 2022?",
    "How do I renew my Emirates ID?",
]
def top_score(q):
    return retrieve(q, top_n=1)[0][1]

print("IN SCOPE (should be HIGH)")
for q in in_scope:     print(f"  {top_score(q):.3f}  {q}")
print("\nOUT OF SCOPE (should be LOW)")
for q in out_of_scope: print(f"  {top_score(q):.3f}  {q}")

IN SCOPE (should be HIGH)
  0.987  How many days of annual leave do I get?
  0.872  What does Article 51 say?
  0.544  My employer has not paid my salary for two months
  1.000  end of service gratuity calculation
  0.827  Can my company share my personal data with others?
  0.994  How long is the probation period?
  0.002  Can my employer keep my passport?

OUT OF SCOPE (should be LOW)
  0.001  What is the fee for a UAE Golden Visa?
  0.000  What is the IRS 401k contribution limit for 2026?
  0.000  Who won the FIFA World Cup in 2022?
  0.000  How do I renew my Emirates ID?


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from src.llm import get_llm
from src.retrieval import rrf

REWRITE_SYSTEM = """You turn a user's question about UAE employment or personal data into a search query
written in the formal vocabulary of UAE legislation.
Examples of word changes: salary -> wage; boss/company -> employer; employee/staff -> worker;
passport/visa papers -> official documents; fired -> termination of employment; vacation -> annual leave.
Keep the meaning. Do not answer the question. Return ONLY the rewritten query on one line."""

rewrite_chain = (
    ChatPromptTemplate.from_messages([("system", REWRITE_SYSTEM), ("human", "{question}")])
    | get_llm()
    | StrOutputParser()          # AIMessage -> plain string
)

def retrieve_with_rewrite(q, candidates=10, top_n=5):
    q2 = rewrite_chain.invoke({"question": q}).strip()
    pool = rrf([hybrid_search(q, top_n=candidates),        # original wording
                hybrid_search(q2, top_n=candidates)],      # legal wording
               top_n=candidates * 2)
    return q2, rerank(q2, pool, top_n=top_n)               # re-rank against the legal wording

tests = [
    "My employer has not paid my salary for two months",
    "Can my employer keep my passport?",
    "How many days of annual leave do I get?",          # should stay correct
    "What is the fee for a UAE Golden Visa?",           # should stay LOW
]
for q in tests:
    q2, res = retrieve_with_rewrite(q)
    print(f"\nQ : {q}\nQ2: {q2}")
    for d, s in res[:3]:
        print(f"   {s:.3f}  {label(d):28} {d.page_content.splitlines()[2]!r}")

INFO:httpx2:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"



Q : My employer has not paid my salary for two months
Q2: The employer has not paid the worker's wage for two months.
   0.935  labour_law:9                 'stipulated in Clause (3) of this Article, unless agreed otherwise bet ween the Worker and'
   0.892  labour_law:47                "Unlawful Termination of Worker's Service"
   0.730  labour_law:29                '1. The Worker may, upon the approval of the Employer and in accordance with the'


INFO:httpx2:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"



Q : Can my employer keep my passport?
Q2: May an employer retain the official documents of a worker?
   0.998  labour_law:13                "Employer's Obligations"
   0.901  labour_law:16                "Worker's Obligations"
   0.636  labour_law:45                'Cases of Worker Leaving Work Without Warning'


INFO:httpx2:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"



Q : How many days of annual leave do I get?
Q2: What is the number of days of annual leave to which a worker is entitled under the UAE Labour Law?
   0.996  labour_law:29                'Annual leave'
   0.973  labour_law:29                '1. The Worker may, upon the approval of the Employer and in accordance with the'
   0.970  labour_law:32                'Various holidays'


INFO:httpx2:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"



Q : What is the fee for a UAE Golden Visa?
Q2: What is the fee for the issuance of a Golden Visa in the UAE?
   0.004  exec_regs:6                  'g. Work Permit for Juvenile: The permit under which the person who has completed 15'
   0.000  pdpl:31                      'Publication & Enforcement of this Decree by Law'
   0.000  labour_law:74                'Publication & Enforcement of this Decree by law'


In [7]:
import logging; logging.getLogger("httpx").setLevel(logging.WARNING)   # hide the INFO:httpx request lines

q  = "My employer has not paid my salary for two months"
q2 = rewrite_chain.invoke({"question": q}).strip()
for n in [10, 20]:
    pool = rrf([hybrid_search(q, k=n, top_n=n), hybrid_search(q2, k=n, top_n=n)], top_n=n * 2)
    ranked = rerank(q2, pool, top_n=len(pool))
    hit = next(((i, s) for i, (d, s) in enumerate(ranked, 1) if label(d) == "labour_law:22"), None)
    print(f"candidates={n:2}: pool {len(pool):2} chunks | Article 22 after rerank: {hit or 'NOT IN POOL'}")

print("\nTop 3 with 20 candidates:", [f"{label(d)} ({s:.2f})" for d, s in ranked[:3]])

INFO:httpx2:HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"


candidates=10: pool 13 chunks | Article 22 after rerank: NOT IN POOL
candidates=20: pool 27 chunks | Article 22 after rerank: NOT IN POOL

Top 3 with 20 candidates: ['labour_law:9 (0.94)', 'labour_law:47 (0.89)', 'labour_law:29 (0.74)']
